# Initial LLM Explainer Prompt — CMA-ES

This notebook contains only the material required to reproduce the **Initial Explainer (Cycle 0)** prompt used in the framework.

The Initial Explainer receives four inputs:

1. **BASE_CMAES_ALGORITHM** — the fixed baseline CMA-ES implementation.
2. **EVALUATED_CMAES_ALGORITHM** — the algorithm evaluated in Cycle 0 (the baseline itself).
3. **CYCLE 0 RESULTS** — the experimental BBOB results obtained for the baseline.
4. **TEN KNOWN CMA-ES REFERENCE MECHANISMS** — conceptual reference mechanisms used only as design knowledge and novelty constraints.

No API call is included in this notebook. Its purpose is to document the exact prompt construction and its inputs.


## 1. Baseline CMA-ES

The complete baseline implementation is supplied to the Initial Explainer as `BASE_CMAES_ALGORITHM`. In Cycle 0, the evaluated algorithm is the same baseline implementation.


In [ ]:
base_cmaes_code = r"""
import numpy as np
import ioh


class CMAES:
    '''
    Baseline CMA-ES.
    '''
    def __init__(self,
                 pop_size: int = None,
                 num_iterations: int = 1000,
                 sigma: float = 0.5):

        self.pop_size = pop_size
        self.num_iterations = num_iterations
        self.sigma = sigma

    def __call__(self, problem: ioh.problem.RealSingleObjective):

        dim = problem.meta_data.n_variables
        low_b, high_b = problem.bounds.lb[0], problem.bounds.ub[0]

        if self.pop_size is None:
            lambda_ = 4 + int(3 * np.log(dim))
        else:
            lambda_ = self.pop_size

        mu = lambda_ // 2

        weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
        weights = weights / np.sum(weights)

        mu_eff = 1.0 / np.sum(weights ** 2)

        c_sigma = (mu_eff + 2) / (dim + mu_eff + 5)

        d_sigma = (
            1
            + 2 * max(0, np.sqrt((mu_eff - 1) / (dim + 1)) - 1)
            + c_sigma
        )

        c_c = (4 + mu_eff / dim) / (dim + 4 + 2 * mu_eff / dim)

        c1 = 2 / ((dim + 1.3) ** 2 + mu_eff)

        c_mu = min(
            1 - c1,
            2 * (mu_eff - 2 + 1 / mu_eff) / ((dim + 2) ** 2 + mu_eff)
        )

        expected_norm = np.sqrt(dim) * (
            1 - 1 / (4 * dim) + 1 / (21 * dim ** 2)
        )

        mean = np.random.uniform(low_b, high_b, dim)
        sigma = self.sigma * (high_b - low_b)

        C = np.eye(dim)

        p_sigma = np.zeros(dim)
        p_c = np.zeros(dim)

        best_solution = None
        best_fitness = np.inf

        history = {
            "best_fitness": [],
            "sigma": [],
            "mean": [],
            "stagnation_counter": 0,
            "restart_count": 0
        }

        for iteration in range(self.num_iterations):

            # Numerical safeguard for covariance matrix
            C = (C + C.T) / 2

            if not np.all(np.isfinite(C)):
                C = np.eye(dim)

            C = C + 1e-12 * np.eye(dim)

            eigenvalues, B = np.linalg.eigh(C)
            eigenvalues = np.maximum(eigenvalues, 1e-20)

            D = np.sqrt(eigenvalues)
            BD = B @ np.diag(D)

            # ========================================================
            # LLM SECTION 1: Sampling
            # ========================================================

            z = np.random.randn(lambda_, dim)

            population = mean + sigma * (z @ BD.T)

            # Numerical safeguard for candidate solutions
            if not np.all(np.isfinite(population)):
                population = np.nan_to_num(
                    population,
                    nan=0.0,
                    posinf=high_b,
                    neginf=low_b
                )

            population = np.clip(population, low_b, high_b)

            # ========================================================
            # END LLM SECTION 1
            # ========================================================

            fitness = np.array([problem(ind) for ind in population])

            sorted_indices = np.argsort(fitness)

            population = population[sorted_indices]
            z = z[sorted_indices]
            fitness = fitness[sorted_indices]

            previous_best_fitness = best_fitness

            if fitness[0] < best_fitness:
                best_fitness = fitness[0]
                best_solution = population[0].copy()
                history["stagnation_counter"] = 0
            else:
                history["stagnation_counter"] += 1

            selected_population = population[:mu]
            selected_z = z[:mu]

            old_mean = mean.copy()

            # ========================================================
            # LLM SECTION 2: Mean update
            # ========================================================

            mean = np.sum(
                weights[:, np.newaxis] * selected_population,
                axis=0
            )

            mean = np.clip(mean, low_b, high_b)

            # ========================================================
            # END LLM SECTION 2
            # ========================================================

            # ========================================================
            # LLM SECTION 3: Adaptation
            # ========================================================

            C_inv_sqrt = B @ np.diag(1 / D) @ B.T

            p_sigma = (
                (1 - c_sigma) * p_sigma
                + np.sqrt(c_sigma * (2 - c_sigma) * mu_eff)
                * (C_inv_sqrt @ ((mean - old_mean) / sigma))
            )

            sigma_exponent = (
                (c_sigma / d_sigma)
                * (np.linalg.norm(p_sigma) / expected_norm - 1)
            )

            # Prevent overflow in the exponential step-size update
            sigma_exponent = np.clip(
                sigma_exponent,
                -50.0,
                50.0
            )

            sigma = sigma * np.exp(sigma_exponent)

            # Recover from numerical instability if sigma becomes NaN or Inf
            if not np.isfinite(sigma):
                sigma = self.sigma * (high_b - low_b)

            sigma = np.clip(
                sigma,
                1e-12,
                1e3 * (high_b - low_b)
            )

            norm_p_sigma = np.linalg.norm(p_sigma)

            h_sigma_condition = norm_p_sigma / np.sqrt(
                1 - (1 - c_sigma) ** (2 * (iteration + 1))
            ) < (1.4 + 2 / (dim + 1)) * expected_norm

            h_sigma = 1.0 if h_sigma_condition else 0.0

            p_c = (
                (1 - c_c) * p_c
                + h_sigma
                * np.sqrt(c_c * (2 - c_c) * mu_eff)
                * ((mean - old_mean) / sigma)
            )

            y = (selected_population - old_mean) / sigma

            rank_mu_update = np.zeros((dim, dim))

            for i in range(mu):
                rank_mu_update += weights[i] * np.outer(y[i], y[i])

            C = (
                (1 - c1 - c_mu) * C
                + c1 * np.outer(p_c, p_c)
                + c_mu * rank_mu_update
            )

            C = (C + C.T) / 2

            # Final covariance safeguard
            if not np.all(np.isfinite(C)):
                C = np.eye(dim)

            C = C + 1e-12 * np.eye(dim)

            # ========================================================
            # END LLM SECTION 3
            # ========================================================

            history["best_fitness"].append(best_fitness)
            history["sigma"].append(sigma)
            history["mean"].append(mean.copy())

        return best_solution, best_fitness
"""

## 2. Reference CMA-ES mechanisms

The following ten mechanisms are provided to the Initial Explainer as conceptual references. They are **not candidate algorithms to copy**. The prompt explicitly requires the LLM to compare its proposal against these references and reject near-copies or substantially equivalent mechanisms.


In [ ]:

# ============================================================
# TEN KNOWN CMA-ES VARIANTS USED AS DESIGN REFERENCES
# Conceptual references only; they are NOT used to replace the
# user's INITIAL CMA-ES implementation.
# ============================================================

CMAES_REFERENCE_VARIANTS = {
    "MIRRORED SAMPLING": {
        "primary_section": "Sampling",
        'code': '# Extracted directly from the user-provided ModEA source.\n# Variant/reference: Mirrored Sampling CMA-ES\n# This file is intended as visible reference code for an LLM prompt; it is not a standalone executable module.\n\n\n# ===== MirroredSampling | source: modea/Sampling.py, lines 211-257 =====\nclass MirroredSampling(object):\n    """\n        A sampler to create mirrored samples using some base sampler (Gaussian by default)\n        Returns a single vector each time, while remembering the internal state of whether the ``next()`` should return\n        a new sample, or the mirror of the previous one.\n\n        :param n:               Dimensionality of the vectors to be sampled\n        :param shape:           String to select between whether column (``\'col\'``) or row (``\'row\'``) vectors should be\n                                returned. Defaults to column vectors\n        :param base_sampler:    A different Sampling object from which samples to be mirrored are drawn. If no\n                                base_sampler is given, a :class:`~GaussianSampling` object will be\n                                created and used.\n    """\n    def __init__(self, n, shape=\'col\', base_sampler=None):\n        self.n = n\n        self.shape = (n,1) if shape == \'col\' else (1,n)\n        self.mirror_next = False\n        self.last_sample = None\n        if base_sampler is None:\n            self.base_sampler = GaussianSampling(n, shape)\n        else:\n            self.base_sampler = base_sampler\n\n    def next(self):\n        """\n            Draw the next sample from the Sampler\n\n            :return:    A new vector, alternating between a new sample from the base_sampler and a mirror of the last.\n        """\n        mirror_next = self.mirror_next\n        self.mirror_next = not mirror_next\n\n        if mirror_next:\n            sample = self.last_sample * -1\n        else:\n            sample = self.base_sampler.next()\n            self.last_sample = sample\n\n        return sample\n\n    def reset(self):\n        """\n            Reset the internal state of this sampler, so the next sample is forced to be taken new.\n        """\n        self.mirror_next = False\n        self.last_sample = None\n        self.base_sampler.reset()\n\n'

    },
    "ORTHOGONAL SAMPLING": {
        "primary_section": "Sampling",
        'code': '# Extracted directly from the user-provided ModEA source.\n# Variant/reference: Orthogonal Sampling CMA-ES\n# This file is intended as visible reference code for an LLM prompt; it is not a standalone executable module.\n\n\n# ===== OrthogonalSampling | source: modea/Sampling.py, lines 120-208 =====\nclass OrthogonalSampling(object):\n    """\n        A sampler to create orthogonal samples using some base sampler (Gaussian as default)\n\n        :param n:               Dimensionality of the vectors to be sampled\n        :param lambda_:         Number of samples to be drawn and orthonormalized per generation\n        :param shape:           String to select between whether column (``\'col\'``) or row (``\'row\'``) vectors should be\n                                returned. Defaults to column vectors\n        :param base_sampler:    A different Sampling object from which samples to be mirrored are drawn. If no\n                                base_sampler is given, a :class:`~GaussianSampling` object will be\n                                created and used.\n    """\n    def __init__(self, n, lambda_, shape=\'col\', base_sampler=None):\n        if n == 0 or lambda_ == 0:\n            raise ValueError("\'n\' ({}) and \'lambda_\' ({}) cannot be zero".format(n, lambda_))\n\n        self.n = n\n        self.shape = (n,1) if shape == \'col\' else (1,n)\n        if base_sampler is None:\n            self.base_sampler = GaussianSampling(n, shape)\n        else:\n            self.base_sampler = base_sampler\n        self.num_samples = int(lambda_)\n        self.current_sample = 0\n        self.samples = None\n\n    def next(self):\n        """\n            Draw the next sample from the Sampler\n\n            :return:    A new vector sampled from a set of orthonormalized vectors, originally drawn from base_sampler\n        """\n        if self.current_sample % self.num_samples == 0:\n            self.current_sample = 0\n            invalid_samples = True\n            while invalid_samples:\n                invalid_samples = self.__generateSamples()\n\n        self.current_sample += 1\n        return self.samples[self.current_sample-1]\n\n    def __generateSamples(self):\n        """ Draw <num_samples> new samples from the base_sampler, orthonormalize them and store to be drawn from """\n        samples = []\n        lengths = []\n        for i in range(self.num_samples):\n            sample = self.base_sampler.next()\n            samples.append(sample)\n            lengths.append(norm(sample))\n\n        num_samples = self.num_samples if self.num_samples <= self.n else self.n\n        samples[:num_samples] = self.__gramSchmidt(samples[:num_samples])\n        for i in range(num_samples):\n            samples[i] *= lengths[i]\n\n        self.samples = samples\n        return any(isnan(samples))  # Are all generated samples any good? I.e. is there no \'nan\' value anywhere?\n\n    def __gramSchmidt(self, vectors):\n        """ Implementation of the Gram-Schmidt process for orthonormalizing a set of vectors """\n        num_vectors = len(vectors)\n        lengths = np.zeros(num_vectors)\n        lengths[0] = norm(vectors[0])\n\n        for i in range(1, num_vectors):\n            for j in range(i):\n                vec_i = vectors[i]\n                vec_j = vectors[j]\n                if lengths[j]: # This will prevent Runtimewarning (Division over zero) \n                    vectors[i] = vec_i - vec_j * (dot(vec_i.T, vec_j) / lengths[j] ** 2)\n            lengths[i] = norm(vectors[i])\n\n        for i, vec in enumerate(vectors):\n            # In the rare, but not uncommon cases of this producing 0-vectors, we simply replace it with a random one\n            if lengths[i] == 0:\n                new_vector = self.base_sampler.next()\n                vectors[i] = new_vector / norm(new_vector)\n            else:\n                vectors[i] = vec / lengths[i]\n\n        return vectors\n\n    def reset(self):\n        """\n            Reset the internal state of this sampler, so the next sample is forced to be taken new.\n        """\n        self.current_sample = 0\n        self.samples = None\n        self.base_sampler.reset()\n\n'
    },
    "QUASI-GAUSSIAN SAMPLING (SOBOL/HALTON)": {
        "primary_section": "Sampling",
        'code': '# Extracted directly from the user-provided ModEA source.\n# Variant/reference: Sobol Quasi-Gaussian CMA-ES\n# This file is intended as visible reference code for an LLM prompt; it is not a standalone executable module.\n\n\n# ===== QuasiGaussianSobolSampling | source: modea/Sampling.py, lines 60-87 =====\nclass QuasiGaussianSobolSampling(object):\n    """\n        A quasi-Gaussian sampler based on a Sobol sequence\n\n        :param n:       Dimensionality of the vectors to be sampled\n        :param shape:   String to select between whether column (``\'col\'``) or row (``\'row\'``) vectors should be\n                        returned. Defaults to column vectors\n    """\n    def __init__(self, n, shape=\'col\', seed=None):\n        self.n = n\n        self.shape = (n,1) if shape == \'col\' else (1,n)\n        if seed is None or seed < 2:\n            self.seed = np.random.randint(2, n**2)  # seed=1 will give a null-vector as first result\n        else:\n            self.seed = seed\n\n    def next(self):\n        """\n            Draw the next sample from the Sampler\n\n            :return:    A new vector sampled from a Sobol sequence with mean 0 and standard deviation 1\n        """\n        vec, seed = i4_sobol(self.n, self.seed)\n        self.seed = seed if seed > 1 else 2\n\n        vec = array(norm_dist.ppf(vec))\n        vec = vec.reshape(self.shape)\n        return vec\n\n\n\n# Extracted directly from the user-provided ModEA source.\n# Variant/reference: Halton Quasi-Gaussian CMA-ES\n# This file is intended as visible reference code for an LLM prompt; it is not a standalone executable module.\n\n\n# ===== QuasiGaussianHaltonSampling | source: modea/Sampling.py, lines 90-117 =====\nclass QuasiGaussianHaltonSampling(object):\n    """\n        A quasi-Gaussian sampler based on a Halton sequence\n\n        :param n:       Dimensionality of the vectors to be sampled\n        :param shape:   String to select between whether column (``\'col\'``) or row (``\'row\'``) vectors should be\n                        returned. Defaults to column vectors\n    """\n    def __init__(self, n, shape=\'col\'):\n\n        if not halton_available:\n            raise ImportError("Package \'ghalton\' not found, QuasiGaussianHaltonSampling not available.")\n        self.n = n\n        self.shape = (n,1) if shape == \'col\' else (1,n)\n        self.halton = Halton(n)\n\n\n    def next(self):\n        """\n            Draw the next sample from the Sampler\n\n            :return:    A new vector sampled from a Halton sequence with mean 0 and standard deviation 1\n        """\n        vec = self.halton.get(1)[0]\n\n        vec = array(norm_dist.ppf(vec))\n        vec = vec.reshape(self.shape)\n        return vec\n\n'
    },
    "THRESHOLD CONVERGENCE": {
        "primary_section": "Adaptation",
        'code': '# Extracted directly from the user-provided ModEA source.\n# Variant/reference: Threshold Convergence CMA-ES\n# This file is intended as visible reference code for an LLM prompt; it is not a standalone executable module.\n\n\n# ===== Threshold scaling | source: modea/Mutation.py, lines 66-82 =====\ndef _scaleWithThreshold(mutation_vector, threshold):\n    """\n        Checks if norm(mutation_vector) is at least the given threshold.\n        If not, the vector is mirrored to the other side of the threshold,\n        i.e. scaled to be length: threshold + (threshold - norm(mutation_vector))\n\n        :param mutation_vector:  Mutation vector to be scaled\n        :param threshold:        Minimum length threshold. Vector is scaled if length does not reach threshold\n        :returns:                The threshold-compliant mutation vector\n    """\n\n    length = norm(mutation_vector)\n    if length < threshold:\n        new_length = threshold + (threshold - length)\n        mutation_vector *= (new_length / length)\n\n    return mutation_vector\n\n\n# ===== CMA mutation with threshold option | source: modea/Mutation.py, lines 130-148 =====\ndef CMAMutation(individual, param, sampler, threshold_convergence=False):\n    """\n        CMA mutation: x = x + (sigma * B*D*N(0,I))\n\n        :param individual:              :class:`~modea.Individual.FloatIndividual` to be mutated\n        :param param:                   :class:`~modea.Parameters.Parameters` object to store settings\n        :param sampler:                 :mod:`~modea.Sampling` module from which the random values should be drawn\n        :param threshold_convergence:   Boolean: Should threshold convergence be applied. Default: False\n    """\n\n    individual.last_z = sampler.next()\n\n    if threshold_convergence:\n        individual.last_z = _scaleWithThreshold(individual.last_z, param.threshold)\n\n    individual.mutation_vector = dot(param.B, (param.D * individual.last_z))  # y_k in cmatutorial.pdf)\n    mutation_vector = individual.mutation_vector * param.sigma\n\n    individual.genotype = _keepInBounds(add(individual.genotype, mutation_vector), param.l_bound, param.u_bound)\n\n\n# ===== Threshold schedule update | source: modea/Parameters.py, lines 393-402 =====\n    def updateThreshold(self, t):\n        """\n            Update the threshold that is used to maintain a minimum stepsize.\n            Taken from: Evolution Strategies with Thresheld Convergence (CEC 2015)\n\n            :param t:   Ammount of the evaluation budget spent\n        """\n\n        budget = self.budget\n        self.threshold = self.init_threshold * self.diameter * ((budget-t) / self.budget)**self.decay_factor\n\n'
    },
    "ALTERNATIVE RECOMBINATION WEIGHTS": {
        "primary_section": "Mean update",
        'code': '# Extracted directly from the user-provided ModEA source.\n# Variant/reference: Alternative Recombination Weights CMA-ES\n# Includes default logarithmic, equal 1/n, and geometric 1/2^n weighting options in this ModEA version.\n# This file is intended as visible reference code for an LLM prompt; it is not a standalone executable module.\n\n\n# ===== getWeights | source: modea/Parameters.py, lines 365-390 =====\n    def getWeights(self, weights_option=None):\n        """\n            Defines a list of weights to be used in weighted recombination. Available options are:\n* ``1/n``: Each weight is set to 1/n\n* ``1/2^n``: Each weight is set to 1/2^i + (1/2^n)/mu\n* ``default``: Each weight is set to log((lambda-1)/2) - log(i)\n\n            :param weights_option:  String to indicate which weights should be used.\n            :returns:               Returns a np.array of weights, adding to 1\n        """\n\n        mu = self.mu_int\n\n        if weights_option == \'1/n\':\n            weights = ones((mu, 1)) * (1/mu)\n        elif weights_option == \'1/2^n\':\n            # The idea here is to give weights (1/2, 1/4, ..., 1/2**mu) + (1/2**mu / mu) so it all sums to 1\n            leftover = (1 / (2**mu)) / mu\n            weights = 1 / 2**arange(1, mu+1) + leftover\n            weights.shape = (mu, 1)\n        else:\n            _mu_prime = (self.lambda_-1) / 2.0\n            weights = log(_mu_prime+1.0)-log(arange(1, mu+1)[:, newaxis])\n            weights = weights / sum(weights)\n\n        return weights\n\n'
    },
    "ACTIVE COVARIANCE UPDATE": {
        "primary_section": "Adaptation",
        'code': '# Extracted directly from the user-provided ModEA source.\n# Variant/reference: Active CMA-ES\n# The active branch subtracts covariance information from the least successful offspring when enough offspring are available.\n# This file is intended as visible reference code for an LLM prompt; it is not a standalone executable module.\n\n\n# ===== adaptCovarianceMatrix including active negative update | source: modea/Parameters.py, lines 263-337 =====\n    def adaptCovarianceMatrix(self, evalcount):\n        """\n            Adapt the covariance matrix according to the (Active-)CMA-ES.\n\n            :param evalcount:   Number of evaluations used by the algorithm so far\n        """\n\n        cc, cs, c_1, c_mu, n = self.c_c, self.c_sigma, self.c_1, self.c_mu, self.n\n        wcm, wcm_old, mueff, invsqrt_C = self.wcm, self.wcm_old, self.mu_eff, self.sqrt_C\n        lambda_ = self.lambda_\n\n        self.p_sigma = (1-cs) * self.p_sigma + \\\n                       sqrt(cs*(2-cs)*mueff) * dot(invsqrt_C, (wcm - wcm_old) / self.sigma)\n        power = (2*evalcount/lambda_)\n        if power < 1000:  #TODO: Solve more neatly\n            hsig = sum(self.p_sigma**2)/(1-(1-cs)**power)/n < 2 + 4/(n+1)\n        else:\n            #Prevent underflow error,\n            hsig = sum(self.p_sigma**2)/n < 2 + 4/(n+1)\n        self.p_c = (1-cc) * self.p_c + hsig * sqrt(cc*(2-cc)*mueff) * (wcm - wcm_old) / self.sigma\n        offset = self.offset[:, :self.mu_int]\n\n        # Regular update of C\n        self.C = (1 - c_1 - c_mu) * self.C \\\n                  + c_1 * (outer(self.p_c, self.p_c) + (1-hsig) * cc * (2-cc) * self.C) \\\n                  + c_mu * dot(offset, self.weights*offset.T)\n        if self.active and len(self.all_offspring) >= 2*self.mu_int:  # Active update of C\n            offset_bad = self.offset[:, -self.mu_int:]\n            self.C -= c_mu * dot(offset_bad, self.weights*offset_bad.T)\n\n        # Adapt step size sigma\n        if self.tpa:\n            alpha_act = self.tpa_result * self.alpha\n            alpha_act += self.beta_tpa if self.tpa_result > 1 else 0\n            self.alpha_s += self.c_alpha * (alpha_act - self.alpha_s)\n            self.sigma *= exp(self.alpha_s)\n        else:\n            exponent = (norm(self.p_sigma) / self.chiN - 1) * self.c_sigma / self.damps\n            if exponent < 1000:  #TODO: Solve more neatly\n                self.sigma = self.sigma * exp(exponent)\n            else:\n                self.sigma = self.sigma_mean\n        self.sigma_mean = self.sigma\n\n        ### Update BD ###\n        C = self.C  # lastest setting for\n        C = triu(C) + triu(C, 1).T                  # eigen decomposition\n\n        degenerated = False\n        if any(isinf(C)) > 1:                           # interval\n            degenerated = True\n            # raise Exception("Values in C are infinite")\n        elif not 1e-16 < self.sigma_mean < 1e6:\n            degenerated = True\n        else:\n            try:\n                w, e_vector = eigh(C)\n                e_value = sqrt(list(map(complex, w))).reshape(-1, 1)\n                if any(~isreal(e_value)):\n                    degenerated = True\n                    # raise Exception("Eigenvalues of C are not real")\n                elif any(isinf(e_value)):\n                    degenerated = True\n                    # raise Exception("Eigenvalues of C are infinite")\n                else:\n                    self.D = real(e_value)\n                    self.B = e_vector\n                    self.sqrt_C = dot(e_vector, e_value**-1 * e_vector.T)\n            except LinAlgError as e:\n                # raise Exception(e)\n                print("Restarting, degeneration detected: {}".format(e))\n                degenerated = True\n\n        if degenerated:\n            self.restart()\n\n'
    },
    "TWO-POINT STEP-SIZE ADAPTATION (TPA)": {
        "primary_section": "Adaptation",
        'code': '# Extracted directly from the user-provided ModEA source.\n# Variant/reference: Two-Point Adaptation CMA-ES\n# TPA uses two special offspring and updates sigma based on their comparison.\n# This file is intended as visible reference code for an LLM prompt; it is not a standalone executable module.\n\n\n# ===== TPA update | source: modea/Algorithms.py, lines 116-128 =====\n    def tpaUpdate(self):\n        wcm = self.parameters.wcm\n        tpa_vector = (wcm - self.parameters.wcm_old) * self.parameters.tpa_factor\n\n        tpa_fitness_plus = self.fitnessFunction((wcm + tpa_vector).flatten())\n        tpa_fitness_min = self.fitnessFunction((wcm - tpa_vector).flatten())\n\n        self.used_budget += 2\n        if self.used_budget > self.budget and self.parameters.sequential:\n            self.used_budget = self.budget\n\n        # Is the ideal step size larger (True) or smaller (False)?\n        self.parameters.tpa_result = 1 if tpa_fitness_plus < tpa_fitness_min else -1\n\n\n# ===== Parameter initialization including TPA state | source: modea/Parameters.py, lines 64-184 =====\n    def __init__(self, n, budget, sigma=None,\n                 mu=None, lambda_=None, weights_option=None, l_bound=None, u_bound=None, seq_cutoff=1, wcm=None,\n                 active=False, elitist=False, local_restart=None, sequential=False, tpa=False,\n                 values=None):\n\n        if lambda_ is None:\n            lambda_ = int(4 + floor(3 * log(n)))\n        eff_lambda = lambda_ - 2 if tpa else lambda_\n        if mu is None:\n            mu = 0.5\n        elif mu > lambda_:\n            raise Exception("mu ({}) cannot be greater than lambda ({})".format(mu, lambda_))\n        elif mu >= 1:\n            mu /= lambda_\n        if sigma is None:\n            sigma = 1\n\n        if l_bound is None or not isfinite(l_bound).all():\n            l_bound = ones((n,)) * -5\n        if u_bound is None or not isfinite(u_bound).all():\n            u_bound = ones((n,)) * 5\n\n        if seq_cutoff is None:\n            seq_cutoff = mu * eff_lambda\n        if wcm is None:\n            wcm = (np.random.randn(n,1) * (u_bound - l_bound)) + l_bound\n\n        ### Basic parameters ###\n        self.n = n\n        self.budget = budget\n        self.mu = mu\n        self.lambda_ = lambda_\n        self.eff_lambda = eff_lambda\n        self.l_bound = l_bound\n        self.u_bound = u_bound\n        self.search_space_size = u_bound - l_bound\n        self.sigma = sigma\n        self.sigma_mean = sigma\n        self.active = active\n        self.elitist = elitist\n        self.local_restart = local_restart\n        self.sequential = sequential\n        self.seq_cutoff = seq_cutoff\n        self.tpa = tpa\n        self.weights_option = weights_option\n        self.weights = self.getWeights(weights_option)\n        self.mu_eff = 1 / sum(square(self.weights))\n\n        ### Meta-parameters ###\n        self.N = 10 * self.n\n\n        ### (1+1)-ES ###\n        self.success_history = zeros((self.N, ), dtype=np.int)\n\n        ### CMA-ES ###\n        # Static\n        mu_eff = self.mu_eff  # Local copy\n        self.c_sigma = (mu_eff + 2) / (mu_eff + n + 5)\n        self.c_c = (4 + mu_eff/n) / (n + 4 + 2*mu_eff/n)\n        self.c_1 = 2 / ((n + 1.3)**2 + mu_eff)\n        self.c_mu = min(1-self.c_1, self.alpha_mu*((mu_eff - 2 + 1/mu_eff) / ((n+2)**2 + self.alpha_mu*mu_eff/2)))\n        self.damps = 1 + 2*np.max([0, sqrt((mu_eff-1)/(n+1))-1]) + self.c_sigma\n        self.chiN = n**.5 * (1-1/(4*n)+1/(21*n**2))  # Expected random vector (or something like it)\n\n        # Dynamic\n        self.C = eye(n)       # Covariance matrix\n        self.sqrt_C = eye(n)\n        self.B = eye(n)       # Eigenvectors of C\n        self.D = ones((n,1))  # Diagonal eigenvalues of C\n        self.s_mean = None\n        self.p_sigma = zeros((n,1))\n        self.p_c = zeros((n,1))\n        self.weighted_mutation_vector = zeros((n,1))   # weighted average of the last generation of offset vectors\n        self.y_w_squared = zeros((n,1))\n        self.offspring = None\n        self.offset = None\n        self.all_offspring = None\n        self.wcm = wcm\n        self.wcm_old = None\n\n        ### Threshold Convergence ###\n        # Static\n        self.diameter = sqrt(sum(square(self.search_space_size)))  # Diameter of the search space\n        self.init_threshold = 0.2  # Guess value, not actually mentioned in paper\n        self.decay_factor = 0.995  # Determines curve of the decay. < 1: \'bulges\', > 1: \'hollow\'\n        # Dynamic\n        self.threshold = self.init_threshold * self.diameter * ((1-0) / 1)**self.decay_factor\n\n        ## Active CMA-ES ##\n        if active:\n            self.c_c = 2 / (n+sqrt(2))**2\n\n        ### Two Point Step Size Adaptation ###\n        # Static\n        self.alpha = 0.5\n        self.tpa_factor = 0.5\n        self.beta_tpa = 0\n        self.c_alpha = 0.3\n        # Dynamic\n        self.alpha_s = 0\n        self.tpa_result = None\n\n        ### IPOP ###\n        self.last_pop = None\n        self.lambda_orig = self.lambda_large = self.lambda_small = self.lambda_\n        self.pop_inc_factor = 2\n        self.flat_fitness_index = int(min([ceil(0.1+self.lambda_/4.0), self.mu_int-1]))\n        self.nbin = 10 + int(ceil(30*n/lambda_))\n        self.histfunevals = zeros(self.nbin)\n\n        self.recent_best_fitnesses = []  # Contains the most recent best fitnesses of the 20 most recent generations\n        self.stagnation_list = []        # Contains median fitness of some recent generations\n        self.is_fitness_flat = False  # (effectively) are all fitness values this generation equal?\n\n        self.max_iter = 100 + 50*(n+3)**2 / sqrt(lambda_)\n        self.tolx = 1e-12 * self.sigma\n        self.tolupx = 1e3 * self.sigma\n\n        self.values = values\n        if values:  # Now we\'ve had the default values, we change all values that were passed along\n            self.__init_values(values)\n\n'
    },
    "ELITIST SELECTION": {
        "primary_section": "Mean update",
        'code': '# Extracted directly from the user-provided ModEA source.\n# Variant/reference: Elitist CMA-ES\n# Elitism changes whether previous parents are included in selection.\n# This file is intended as visible reference code for an LLM prompt; it is not a standalone executable module.\n\n\n# ===== best selection with elitist switch | source: modea/Selection.py, lines 35-56 =====\ndef best(population, new_population, param):\n    """\n        Given the population, return the (mu) best. Also performs some \'housekeeping\' for the CMA-ES by collecting\n        all genotypes and most recent mutation vectors and storing them in the ``param`` object.\n\n        :param population:      List of :class:`~modea.Individual.FloatIndividual` objects containing the previous generation\n        :param new_population:  List of :class:`~modea.Individual.FloatIndividual` objects containing the new generation\n        :param param:           :class:`~modea.Parameters.Parameters` object for storing all parameters, options, etc.\n        :returns:               A slice of the sorted new_population list.\n    """\n    if param.elitist:\n        new_population.extend(population)\n\n    new_population.sort(key=Utils.getFitness)  # sort ascending\n\n    # TODO: REMOVE THESE OPERATIONS FROM THIS FUNCTION, UNEXPECTED/UNDOCUMENTED FUNCTIONALITY\n    offspring = np.column_stack([ind.genotype for ind in new_population])  # Update to use the actual mutations\n    offset = np.column_stack([ind.mutation_vector for ind in new_population])\n    param.all_offspring = offspring\n    param.offset = offset\n\n    return new_population[:param.mu_int]\n\n'
    },
    "SEQUENTIAL SELECTION": {
        "primary_section": "Sampling",
        'code': '# Extracted directly from the user-provided ModEA source.\n# Variant/reference: Sequential Selection CMA-ES\n# Sequential evaluation can stop after an improvement once the cutoff is reached.\n# This file is intended as visible reference code for an LLM prompt; it is not a standalone executable module.\n\n\n# ===== Population evaluation and sequential evaluation | source: modea/Algorithms.py, lines 84-113 =====\n    def evalPopulation(self):\n        for ind in self.new_population:\n            self.mutate(ind, self.parameters)\n        fitnesses = self.fitnessFunction([ind.genotype.flatten() for ind in self.new_population])\n        for ind, fit in zip(self.new_population, fitnesses):\n            ind.fitness = fit\n\n        self.used_budget += self.parameters.lambda_\n        self.gen_size = self.parameters.lambda_\n\n\n    def evalPopulationSequentially(self):\n        improvement_found = False\n        self.gen_size = 0\n        for i, individual in enumerate(self.new_population):\n            self.mutate(individual, self.parameters)  # Mutation\n            # Evaluation\n            individual.fitness = self.fitnessFunction(individual.genotype.flatten())\n            self.used_budget += 1\n            self.gen_size += 1\n\n            # Sequential Evaluation\n            if self.parameters.sequential:  # We interrupt once a better individual has been found\n                if individual.fitness < self.best_individual.fitness:\n                    improvement_found = True\n                if i >= self.seq_cutoff and improvement_found:\n                    break\n                if self.used_budget == self.budget:\n                    break\n        self.new_population = self.new_population[:i+1]  # Discard unused individuals\n\n'
    },
    "IPOP/BIPOP RESTARTS": {
        "primary_section": "Adaptation",
        'code': '# Extracted directly from the user-provided ModEA source.\n# Variant/reference: IPOP / BIPOP CMA-ES\n# This file is intended as visible reference code for an LLM prompt; it is not a standalone executable module.\n\n\n# ===== Local restart optimizer and BIPOP regime | source: modea/Algorithms.py, lines 186-257 =====\n    def runLocalRestartOptimizer(self,target=None, threshold=None):\n        """Run the baseAlgorithm with the given specifications using a local-restart strategy."""\n\n        parameter_opts = self.parameters.getParameterOpts()\n\n        if parameter_opts[\'lambda_\']:\n            lambda_init = parameter_opts[\'lambda_\']\n        elif parameter_opts[\'local_restart\'] in [\'IPOP\', \'BIPOP\']:\n            lambda_init = int(4 + floor(3 * log(parameter_opts[\'n\'])))\n        else:\n            lambda_init = None\n        parameter_opts[\'lambda_\'] = lambda_init\n\n        # BIPOP Specific parameters\n        self.lambda_ = {\'small\': None, \'large\': lambda_init}\n        self.budgets = {\'small\': None, \'large\': None}\n        self.regime = \'first\'  # Later alternates between \'large\' and \'small\'\n\n        while self.total_used_budget < self.total_budget:\n\n            # Every local restart needs its own parameters, so parameter update/mutation must also be linked every time\n            self.parameters = Parameters(**parameter_opts)\n            self.seq_cutoff = self.parameters.mu_int * self.parameters.seq_cutoff\n            self.mutateParameters = self.parameters.adaptCovarianceMatrix\n\n            self.initializePopulation()\n            parameter_opts[\'wcm\'] = self.population[0].genotype\n            self.new_population = self.recombine(self.population, self.parameters)\n\n            # Run the actual algorithm\n            self.runOptimizer(target=target,threshold=threshold)\n            self.total_used_budget += self.used_budget\n\n            if target is not None and self.best_individual.fitness - target < threshold:\n                break\n            # Increasing Population Strategies\n            if parameter_opts[\'local_restart\'] == \'IPOP\':\n                parameter_opts[\'lambda_\'] *= 2\n\n            elif parameter_opts[\'local_restart\'] == \'BIPOP\':\n                try:\n                    self.budgets[self.regime] -= self.used_budget\n                    self.determineRegime()\n                except KeyError:  # Setup of the two regimes after running regularily for the first time\n                    remaining_budget = self.total_budget-self.used_budget\n                    self.budgets[\'small\'] = remaining_budget // 2\n                    self.budgets[\'large\'] = remaining_budget - self.budgets[\'small\']\n                    self.regime = \'large\'\n\n                if self.regime == \'large\':\n                    self.lambda_[\'large\'] *= 2\n                    parameter_opts[\'sigma\'] = 2\n                elif self.regime == \'small\':\n                    rand_val = np.random.random() ** 2\n                    self.lambda_[\'small\'] = int(floor(lambda_init * (.5*self.lambda_[\'large\'] / lambda_init)**rand_val))\n                    parameter_opts[\'sigma\'] = 2e-2 * np.random.random()\n\n                self.budget = self.budgets[self.regime]\n                self.used_budget = 0\n                parameter_opts[\'budget\'] = self.budget\n                parameter_opts[\'lambda_\'] = self.lambda_[self.regime]\n\n\n    def determineRegime(self):\n        large = self.budgets[\'large\']\n        small = self.budgets[\'small\']\n        if large <= 0:\n            self.regime = \'small\'\n        elif small <= 0 or large > small:\n            self.regime = \'large\'\n        else:\n            self.regime = \'small\'\n\n\n# ===== Restart and local restart conditions | source: modea/Parameters.py, lines 405-507 =====\n    def restart(self):\n        """\n            Very basic restart, done by resetting some of the variables for CMA-ES\n        """\n\n        n = self.n\n        self.C = eye(n)\n        self.B = eye(n)\n        self.D = ones((n,1))\n        self.p_sigma = zeros((n, 1))\n        self.sigma_mean = self.sigma = 1          # TODO: make this depend on any input default sigma value\n        # TODO: add feedback of resetting sigma to the sigma per individual\n\n    def recordRecentFitnessValues(self, evalcount, fitnesses):\n        """\n            Record recent fitness values at current budget\n        """\n        self.histfunevals[int(mod(evalcount/self.lambda_-1, self.nbin))] = min(fitnesses)\n\n        self.recent_best_fitnesses.append(min(fitnesses))\n        self.recent_best_fitnesses = self.recent_best_fitnesses[-20:]\n\n        self.stagnation_list.append(median(fitnesses))\n        self.stagnation_list = self.stagnation_list[-int(ceil(0.2*evalcount + 120 + 30*self.n/self.lambda_)):]\n\n        flat_fitness_index = min(len(fitnesses)-1, self.flat_fitness_index)\n        self.is_fitness_flat = min(fitnesses) == sorted(fitnesses)[flat_fitness_index]\n\n\n    def checkLocalRestartConditions(self, evalcount):\n        """\n            Check for local restart conditions according to (B)IPOP\n\n            :param evalcount:   Counter for the current generation\n            :returns:           Boolean value ``restart_required``, True if a restart should be performed\n        """\n\n        if not self.local_restart:\n            return False\n\n        debug = False\n\n        restart_required = False\n        diagC = diag(self.C).reshape(-1, 1)\n        tmp = append(abs(self.p_c), sqrt(diagC), axis=1)\n        a = int(mod(evalcount/self.lambda_-1, self.n))\n\n        # TolX\n        if all(self.sigma*(max(tmp, axis=1)) < self.tolx):\n            if debug:\n                print(\'TolX\')\n            restart_required = True\n\n        # TolUPX\n        elif any(self.sigma*sqrt(diagC)) > self.tolupx:\n            if debug:\n                print(\'TolUPX\')\n            restart_required = True\n\n        # No effective axis\n        elif all(0.1*self.sigma*self.D[a, 0]*self.B[:, a] + self.wcm == self.wcm):\n            if debug:\n                print(\'noeffectaxis\')\n            restart_required = True\n\n        # No effective coordinate\n        elif any(0.2*self.sigma*sqrt(diagC) + self.wcm == self.wcm):\n            if debug:\n                print(\'noeffectcoord\')\n            restart_required = True\n\n        # Condition of C\n        elif cond(self.C) > self.conditioncov:\n            if debug:\n                print(\'condcov\')\n            restart_required = True\n\n        elif mod(evalcount, self.lambda_) == self.nbin and \\\n                max(self.histfunevals) - min(self.histfunevals) < self.tolfun:\n            if debug:\n                print(\'tolfun\')\n            restart_required = True\n\n        # Adjust step size in case of equal function values\n        elif self.is_fitness_flat:\n            if debug:\n                print(\'flatfitness\')\n            restart_required = True\n\n        # A mismatch between sigma increase and decrease of all eigenvalues in C\n        elif self.sigma / 1 > self.tolupsigma*max(self.D):\n            if debug:\n                print(\'tolupsigma\')\n            restart_required = True\n\n        # Stagnation, median of most recent 20 best values is no better than that of the oldest 20 medians/generation\n        elif len(self.stagnation_list) > 20 and len(self.recent_best_fitnesses) > 20 and \\\n                median(self.stagnation_list[:20]) > median(self.recent_best_fitnesses):\n            if debug:\n                print(\'stagnation\')\n            restart_required = True\n\n        return restart_required\n\n'
    },
}


def construir_referencias_cmaes_para_prompt():
    bloques = []

    for nombre, info in CMAES_REFERENCE_VARIANTS.items():
        bloques.append(
            f"REFERENCE: {nombre}\n"
            f"PRIMARY SECTION: {info['primary_section']}\n"
            f"CONCEPT: {info['code']}\n"
        )

    return "\n".join(bloques)


CMAES_REFERENCE_VARIANTS_TEXT = construir_referencias_cmaes_para_prompt()


## 3. Cycle 0 experimental results input

The third input is the experimental history/results produced after evaluating the baseline on BBOB. For repository documentation, the variable below is left as an explicit input slot so the prompt structure can be inspected independently of a particular run.


In [ ]:
# Replace this text with the exact Cycle 0 result block from a run when reproducing the prompt.
cycle_0_results = """
<CYCLE 0 BBOB RESULTS>
"""


## 4. Initial Explainer prompt

This is the exact Cycle 0 prompt template extracted from the framework.


In [ ]:
# EXPLAINER PROMPT - CYCLE 0

def cmaes_explainability_prompt_cycle_0(
        base_algorithm: str,
        evaluated_algorithm: str,
        history: str,
        reference_variants: str = CMAES_REFERENCE_VARIANTS_TEXT):

    return f"""
You are an expert in CMA-ES, BBOB benchmarking, and algorithm explainability.

GOAL
Analyze the baseline CMA-ES evaluated in cycle 0 and propose the first novel CMA-ES modification for cycle 1.

INPUTS

(A) BASE_CMAES_ALGORITHM:
{base_algorithm}

(B) EVALUATED_CMAES_ALGORITHM:
{evaluated_algorithm}

(C) CYCLE 0 RESULTS:
{history}


TEN KNOWN CMA-ES REFERENCE MECHANISMS
{reference_variants}

REFERENCE-MECHANISM POLICY

Use the supplied mechanisms as design knowledge only.
Do NOT directly reproduce or lightly rename any of them.

Before proposing TRY, compare the candidate mechanism against all ten references.
Reject the candidate if its core principle is substantially equivalent to one of them.

NEAR-COPY REJECTION RULE

A mechanism is still considered substantially equivalent if it only changes:
- activation frequency,
- trigger timing,
- offspring fraction,
- a scalar multiplier,
- a threshold,
- or a schedule.

Examples:
- periodic/partial Sobol or Halton injection is still quasi-Gaussian low-discrepancy sampling;
- scaled mirrored sampling is still mirrored sampling;
- thresholded negative covariance update is still Active CMA in core principle.

The TRY report must include:
- Difference from supplied references: explain why the proposal is not substantially equivalent.

CONTEXT
- Cycle 0 corresponds to the fixed baseline CMA-ES.
- There is no previous LLM-generated variant.
- BBOB values represent error to the known optimum.
- Lower error is better.
- Error close to zero indicates that the optimum was reached.

TASK

Internally analyze:
1. Sampling.
2. Mean update.
3. Adaptation (step-size sigma and covariance matrix C).
4. The BBOB performance of the baseline.
5. The main weaknesses or opportunities for improvement.

REFERENCE SPACE

Use known CMA-ES mechanisms and variants only as inspiration.
You may consider ideas related to:

- alternative sampling strategies,
- modified mean recombination,
- step-size adaptation,
- covariance adaptation,
- covariance regularization,
- restart strategies,
- population-size strategies,
- boundary-handling strategies.

Do not simply copy or reproduce an existing CMA-ES variant.

Based on the baseline behavior and the reference mechanisms,
propose ONE new, small, and interpretable algorithmic modification
for cycle 1.

The proposal may adapt, transform, or combine ideas from known
CMA-ES mechanisms, but it must remain simple enough to isolate
and explain its effect experimentally.

OUTPUT FORMAT

Return only:
PROPOSE:
- Mechanism to modify: ...
- Proposed modification: ...
- Inspiration: ...
- Expected purpose: ...

RULES

- Propose only ONE main modification.
- The proposal must not simply reproduce an existing CMA-ES variant.
- Keep the modification small and interpretable.
- Do not include the full internal analysis.
- Do not generate Python code.
- Do not use markdown tables.
"""


## 5. Construct the complete Initial Explainer input

At Cycle 0, both `base_algorithm` and `evaluated_algorithm` contain the fixed baseline CMA-ES. The reference mechanisms are inserted automatically through `CMAES_REFERENCE_VARIANTS_TEXT`.


In [ ]:
# Cycle 0 input mapping used by the framework
base_algorithm = base_cmaes_code
evaluated_algorithm = base_cmaes_code
history = cycle_0_results

initial_explainer_prompt = cmaes_explainability_prompt_cycle_0(
    base_algorithm=base_algorithm,
    evaluated_algorithm=evaluated_algorithm,
    history=history,
    reference_variants=CMAES_REFERENCE_VARIANTS_TEXT
)

# Uncomment to inspect the complete prompt exactly as it would be sent to the LLM.
# print(initial_explainer_prompt)


## Prompt output contract

The Initial Explainer is instructed to return only:

```text
PROPOSE:
- Mechanism to modify: ...
- Proposed modification: ...
- Inspiration: ...
- Expected purpose: ...
```

The proposal must contain only **one main conceptual modification**, remain small and interpretable, and must not directly reproduce any of the ten supplied CMA-ES reference mechanisms.
